# 04 — Deployment and Inference
### Getting the pipeline out of your laptop

---

**Format:** Narrated demo — pre-run outputs. Read the commands, understand the choices.

**Two things to deploy, two independent choices:**

```
┌─────────────────────────────────────────────────────────────────┐
│  CHOICE 1 — Where does the Metaflow flow run?                   │
│                                                                 │
│  Outerbounds       managed Kubernetes + Argo, zero infra ops    │
│  AWS (EKS)         same commands, your cloud account            │
│  Linux box + cron  simplest path, no Kubernetes required        │
│                                                                 │
│  CHOICE 2 — What LLM endpoint do the agents call?               │
│                                                                 │
│  AI Navigator      local, no API key, dev only                  │
│  vLLM              self-hosted GPU, maximum throughput          │
│  Anaconda Platform managed, governed, enterprise                │
└─────────────────────────────────────────────────────────────────┘
```

These choices are independent. Any flow target works with any inference target. The only wiring between them is environment variables.

**The flow code from Module 03 doesn't change for any combination.**

---
# Part 1 — Where the flow runs

## Option A — Outerbounds

Managed Metaflow. You get the full stack — Kubernetes, Argo Workflows, metadata service, artifact storage, Metaflow UI — without operating any of it. Built by the Metaflow team.

In [ ]:
# One-time setup — sign up at outerbounds.com, then:
#
#   pip install outerbounds
#   outerbounds workspaces pull
#
# That writes ~/.metaflowconfig/config.json — Metaflow now points at Outerbounds.
# Nothing else changes.

print("""Metaflow config written to ~/.metaflowconfig/config.json
Connected to Outerbounds workspace: anaconda-demo
Metadata service: https://api.perimeter.outerbounds.co/...
Artifact store:   s3://ob-anaconda-demo-artifacts/""")

Metaflow config written to ~/.metaflowconfig/config.json
Connected to Outerbounds workspace: anaconda-demo
Metadata service: https://api.perimeter.outerbounds.co/...
Artifact store:   s3://ob-anaconda-demo-artifacts/


In [ ]:
# Deploy — identical command regardless of which inference endpoint is configured
# python flows/lightcurve_flow.py --with retry argo-workflows create
#
# --with retry adds @retry to every step for transient failures.
# Per-step @conda environments from Module 03 are respected automatically —
# Outerbounds creates isolated containers per step.

print("""Workflow template 'lightcurve-analysis-flow' created on Argo Workflows.
View in Metaflow UI: https://ui.outerbounds.com/anaconda-demo/LightcurveAnalysisFlow

Trigger:   python flows/lightcurve_flow.py argo-workflows trigger --targets wasp18b
Schedule:  python flows/lightcurve_flow.py argo-workflows create --schedule '0 */6 * * *'
Inspect:   python flows/lightcurve_flow.py argo-workflows status""")

Workflow template 'lightcurve-analysis-flow' created on Argo Workflows.
View in Metaflow UI: https://ui.outerbounds.com/anaconda-demo/LightcurveAnalysisFlow

Trigger:   python flows/lightcurve_flow.py argo-workflows trigger --targets wasp18b
Schedule:  python flows/lightcurve_flow.py argo-workflows create --schedule '0 */6 * * *'
Inspect:   python flows/lightcurve_flow.py argo-workflows status


---
## Option B — AWS (EKS + Argo Workflows)

Same commands as Outerbounds once configured. One-time setup per team. Outerbounds publishes the full [Terraform setup guide](https://docs.outerbounds.com/engineering/deployment/aws-k8s/deployment/).

In [ ]:
# One-time config — replaces outerbounds workspaces pull
# After this, the deploy command is identical

print("""$ metaflow configure aws

  METAFLOW_DATASTORE_SYSROOT_S3    s3://your-bucket/metaflow
  METAFLOW_DEFAULT_DATASTORE       s3
  METAFLOW_DEFAULT_METADATA        service
  METAFLOW_KUBERNETES_NAMESPACE    metaflow

Done. Deploy command is now identical to Outerbounds:
  python flows/lightcurve_flow.py --with retry argo-workflows create

Want to validate before standing up EKS?
  metaflow-dev up       # local Kubernetes + Argo + Metaflow UI via Minikube
  metaflow-dev shell    # shell with Metaflow pointed at local stack""")

$ metaflow configure aws

  METAFLOW_DATASTORE_SYSROOT_S3    s3://your-bucket/metaflow
  METAFLOW_DEFAULT_DATASTORE       s3
  METAFLOW_DEFAULT_METADATA        service
  METAFLOW_KUBERNETES_NAMESPACE    metaflow

Done. Deploy command is now identical to Outerbounds:
  python flows/lightcurve_flow.py --with retry argo-workflows create

Want to validate before standing up EKS?
  metaflow-dev up       # local Kubernetes + Argo + Metaflow UI via Minikube
  metaflow-dev shell    # shell with Metaflow pointed at local stack


---
## Option C — Linux box + cron

No Kubernetes. No Argo. A VM, conda, and a cron job. Right for simple workloads where multi-machine parallelism isn't needed. Fastest to set up.

**What you give up:** multi-machine `foreach` parallelism, the Metaflow UI, event triggering.  
**What you keep:** `@conda` per step, `@retry`, full artifact versioning, Client API inspection.

In [ ]:
# Full setup sequence — runs on the target Linux box, not here
# Use absolute paths in the cron entry — cron doesn't inherit your shell PATH

print("""# On the Linux box (EC2 t3.medium, Render, PythonAnywhere, any VM)

# 1. Install Miniconda
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b && source ~/.bashrc

# 2. Clone and set up
git clone https://github.com/Anaconda-Sandbox/building-intelligent-apps-with-anaconda
cd building-intelligent-apps-with-anaconda/03-multi-agent-architecture
conda env create -f environment.yml && conda activate multi-agent

# 3. Set inference credentials (see Part 2 below for options)
export INFERENCE_BASE_URL="https://api.anthropic.com"
export ANTHROPIC_API_KEY="your_key"

# 4. Test it runs
python flows/lightcurve_flow.py run --targets wasp18b

# 5. Schedule with cron (crontab -e)
0 */6 * * * /home/ubuntu/miniconda3/envs/multi-agent/bin/python \\
    /home/ubuntu/.../flows/lightcurve_flow.py \\
    run --targets wasp18b,wasp12b >> /var/log/lightcurve_flow.log 2>&1""")

# On the Linux box (EC2 t3.medium, Render, PythonAnywhere, any VM)

# 1. Install Miniconda
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b && source ~/.bashrc

# 2. Clone and set up
git clone https://github.com/Anaconda-Sandbox/building-intelligent-apps-with-anaconda
cd building-intelligent-apps-with-anaconda/03-multi-agent-architecture
conda env create -f environment.yml && conda activate multi-agent

# 3. Set inference credentials (see Part 2 below for options)
export INFERENCE_BASE_URL="https://api.anthropic.com"
export ANTHROPIC_API_KEY="your_key"

# 4. Test it runs
python flows/lightcurve_flow.py run --targets wasp18b

# 5. Schedule with cron (crontab -e)
0 */6 * * * /home/ubuntu/miniconda3/envs/multi-agent/bin/python \
    /home/ubuntu/.../flows/lightcurve_flow.py \
    run --targets wasp18b,wasp12b >> /var/log/lightcurve_flow.log 2>&1


---
## Flow deployment — comparison

| | Outerbounds | AWS (EKS) | Linux + cron |
|---|---|---|---|
| Setup time | ~5 min | Hours–days | ~15 min |
| Infra to operate | None | EKS + Argo | One VM |
| `foreach` parallelism | Multi-machine | Multi-machine | Single machine |
| Metaflow UI | ✓ | ✓ | ✗ |
| Event triggering | ✓ | ✓ | ✗ |
| `@conda` per step | ✓ | ✓ | ✓ |
| `@retry` | ✓ | ✓ | ✓ |
| Artifact versioning | S3 managed | S3 yours | `~/.metaflow/` |
| Flow code changes | **None** | **None** | **None** |

---
# Part 2 — What the agents call

Regardless of where the flow runs, the agents inside it need an LLM endpoint. All three options below expose an OpenAI-compatible `/v1/chat/completions` API. The agents call them identically.

```python
# inference_client.py — the same client across all three targets
from openai import OpenAI
import os

client = OpenAI(
    base_url=os.environ["INFERENCE_BASE_URL"],
    api_key=os.environ["INFERENCE_API_KEY"],
)
```

Set the env vars before running the flow. Nothing in the flow code changes.

---
## Inference Target 1 — AI Navigator (local)

AI Navigator is Anaconda's desktop app for running open-source LLMs locally. No API key, no cloud, no network dependency. Right for development on your laptop.

**Limitation:** Single-user, serial requests. The `foreach` parallelism in the flow queues requests rather than running them concurrently. Fine for dev, not for production.

1. Open AI Navigator → Models → download any **Desktop Deployable** model
2. API Server tab → select model → Start
3. Set env vars:

In [ ]:
import os
import sys
sys.path.insert(0, ".")

# Set env vars for AI Navigator
os.environ["INFERENCE_BASE_URL"] = "http://localhost:8080/v1"
os.environ["INFERENCE_API_KEY"]  = "not-needed"
os.environ["INFERENCE_MODEL"]    = ""

from inference_client import check_connection

# Pre-run output shown — run live if AI Navigator is open
print("""Inference target: AI Navigator
  base_url : http://localhost:8080/v1
  api_key  : (any string, or empty if no key set)
  model    : (empty — model is selected in the UI)

Connection check:
  status   : ok
  response : ready
  latency  : 0.84s

Note: use /v1 suffix — the openai client needs the OpenAI-compatible endpoint,
not the native llama.cpp /completion endpoint at localhost:8080.""")

Inference target: AI Navigator
  base_url : http://localhost:8080/v1
  api_key  : (any string, or empty if no key set)
  model    : (empty — model is selected in the UI)

Connection check:
  status   : ok
  response : ready
  latency  : 0.84s

Note: use /v1 suffix — the openai client needs the OpenAI-compatible endpoint,
not the native llama.cpp /completion endpoint at localhost:8080.


---
## Inference Target 2 — vLLM (self-hosted GPU)

vLLM serves HuggingFace models on NVIDIA GPUs via an OpenAI-compatible API. Continuous batching handles concurrent requests efficiently — the `foreach` parallelism in the flow works properly here.

Start the server on your GPU box (see `deploy/vllm.md` for full setup):

```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-Instruct-v0.3 \
    --host 0.0.0.0 --port 8000
```

Then on the calling machine:

In [ ]:
os.environ["INFERENCE_BASE_URL"] = "http://your-gpu-server:8000/v1"
os.environ["INFERENCE_API_KEY"]  = "not-needed"
os.environ["INFERENCE_MODEL"]    = "mistralai/Mistral-7B-Instruct-v0.3"

# Pre-run output:
print("""Inference target: vLLM
  base_url : http://your-gpu-server:8000/v1
  api_key  : not-needed (add --api-key to vLLM startup for auth)
  model    : mistralai/Mistral-7B-Instruct-v0.3

Connection check:
  status   : ok
  response : ready
  latency  : 0.31s

vLLM notes:
  - HuggingFace safetensors format — not the same files as AI Navigator GGUF
  - Module 05 uses Nemotron 3 Nano here instead of Mistral
  - /metrics endpoint available for Prometheus throughput monitoring""")

Inference target: vLLM
  base_url : http://your-gpu-server:8000/v1
  api_key  : not-needed (add --api-key to vLLM startup for auth)
  model    : mistralai/Mistral-7B-Instruct-v0.3

Connection check:
  status   : ok
  response : ready
  latency  : 0.31s

vLLM notes:
  - HuggingFace safetensors format — not the same files as AI Navigator GGUF
  - Module 05 uses Nemotron 3 Nano here instead of Mistral
  - /metrics endpoint available for Prometheus throughput monitoring


---
## Inference Target 3 — Anaconda Platform Model Servers

Anaconda Platform (self-hosted, v7.0.0+) manages model serving at the org level: curated catalog, benchmark-based model selection, governance policies, AIBOM download, Responsible AI scoring (OWASP, MITRE ATLAS via Gray Swan).

1. Admin creates a Model Server from the Model Catalog
2. Copy the server address from the server details page
3. Generate an API key from the API Keys page
4. Set env vars:

In [ ]:
os.environ["INFERENCE_BASE_URL"] = os.environ.get("MODEL_SERVER_BASE_URL", "http://your-platform-server:port")
os.environ["INFERENCE_API_KEY"]  = os.environ.get("ANACONDA_API_KEY",      "your-api-key")
os.environ["INFERENCE_MODEL"]    = ""   # model is loaded on the server

# Pre-run output:
print("""Inference target: Anaconda Platform Model Server
  base_url : http://your-platform-server:port
  api_key  : (from Platform API Keys page)
  model    : (empty string — model is already loaded on the server)

Connection check:
  status   : ok
  response : ready
  latency  : 0.42s

Platform notes:
  - Model Governance controls which models your team can use
  - AIBOM available per model: SHA-256 hashes, benchmark scores, ethical considerations
  - Responsible AI tab: OWASP LLM Top 10, MITRE ATLAS, Gray Swan robustness score
  - Model Catalog chart view: compare TruthfulQA score vs RAM for model selection
  - Model parameter passed as empty string — server ignores it""")

Inference target: Anaconda Platform Model Server
  base_url : http://your-platform-server:port
  api_key  : (from Platform API Keys page)
  model    : (empty string — model is already loaded on the server)

Connection check:
  status   : ok
  response : ready
  latency  : 0.42s

Platform notes:
  - Model Governance controls which models your team can use
  - AIBOM available per model: SHA-256 hashes, benchmark scores, ethical considerations
  - Responsible AI tab: OWASP LLM Top 10, MITRE ATLAS, Gray Swan robustness score
  - Model Catalog chart view: compare TruthfulQA score vs RAM for model selection
  - Model parameter passed as empty string — server ignores it


---
# Part 3 — The portable contract

Three flow targets × three inference targets = nine combinations. All work. None require code changes.

The only wiring is two env vars:

In [ ]:
# Show a complete production combination — Outerbounds orchestration + Platform inference

print("""Combination: Outerbounds + Anaconda Platform

  # Config (one-time)
  outerbounds workspaces pull
  export INFERENCE_BASE_URL="$MODEL_SERVER_BASE_URL"
  export INFERENCE_API_KEY="$ANACONDA_API_KEY"
  export INFERENCE_MODEL=""

  # Deploy
  python flows/lightcurve_flow.py --with retry argo-workflows create

  # Trigger
  python flows/lightcurve_flow.py argo-workflows trigger --targets wasp18b,wasp12b

  # Inspect last run
  from metaflow import Flow
  run = Flow('LightcurveAnalysisFlow').latest_run
  print(run.data.all_results)

Flow code: unchanged from Module 03.
Agent code: unchanged from Module 03.
ingestion.py: unchanged from Module 01.""")

Combination: Outerbounds + Anaconda Platform

  # Config (one-time)
  outerbounds workspaces pull
  export INFERENCE_BASE_URL="$MODEL_SERVER_BASE_URL"
  export INFERENCE_API_KEY="$ANACONDA_API_KEY"
  export INFERENCE_MODEL=""

  # Deploy
  python flows/lightcurve_flow.py --with retry argo-workflows create

  # Trigger
  python flows/lightcurve_flow.py argo-workflows trigger --targets wasp18b,wasp12b

  # Inspect last run
  from metaflow import Flow
  run = Flow('LightcurveAnalysisFlow').latest_run
  print(run.data.all_results)

Flow code: unchanged from Module 03.
Agent code: unchanged from Module 03.
ingestion.py: unchanged from Module 01.


---
## Full comparison

**Flow orchestration:**

| | Outerbounds | AWS (EKS) | Linux + cron |
|---|---|---|---|
| Setup time | ~5 min | Hours–days | ~15 min |
| Infra to operate | None | EKS + Argo | One VM |
| `foreach` parallelism | Multi-machine | Multi-machine | Single machine |
| Metaflow UI | ✓ | ✓ | ✗ |
| Event triggering | ✓ | ✓ | ✗ |
| `@conda` per step | ✓ | ✓ | ✓ |
| Flow code changes | **None** | **None** | **None** |

**Inference endpoints:**

| | AI Navigator | vLLM | Anaconda Platform |
|---|---|---|---|
| Hardware | CPU + RAM/VRAM | GPU required | Platform-managed |
| Concurrency | Serial (slots) | Continuous batching | Managed |
| Model format | GGUF | HuggingFace safetensors | Platform catalog |
| Governance | None | None | Model Governance + AIBOM |
| Auth | Optional API key | Optional `--api-key` | Mandatory API key |
| Agent code changes | **None** | **None** | **None** |

---

**Next:** [Module 05 — GPU-Accelerated Intelligence](../05-gpu-accelerated-intelligence/) — the same pipeline, moved to Nemotron on vLLM on a Brev GPU instance. CUDA Python feature engineering replaces the CPU Polars path. The flow code still doesn't change.